In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [11]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [15]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_93686/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Jaylen Brown,Over,26.5,-137,2025-11-16,2025-11-16T19:08:02Z
1,Underdog,player_points,Jaylen Brown,Under,26.5,-137,2025-11-16,2025-11-16T19:08:02Z
2,Underdog,player_points,Anfernee Simons,Over,14.5,-137,2025-11-16,2025-11-16T19:08:02Z
3,Underdog,player_points,Anfernee Simons,Under,14.5,-137,2025-11-16,2025-11-16T19:08:02Z
4,Underdog,player_points,James Harden,Over,23.5,-137,2025-11-16,2025-11-16T19:08:02Z


### Update projected starting lineups

In [16]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Applications/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


### Top EVs for single bets

In [17]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 113 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Dillon Brooks,Bovada,20.5,23.59,Over,205,1,10.30,103.0,0.502,High
1,Dillon Brooks,Bovada,19.5,23.59,Over,165,1,8.79,87.9,0.533,High
2,Isaac Okoro,Bovada,9.5,10.67,Over,200,0,8.02,80.2,0.401,High
3,Lauri Markkanen,Bovada,30.5,32.72,Over,180,0,7.64,76.4,0.424,High
4,Dillon Brooks,Bovada,18.5,23.59,Over,130,1,7.48,74.8,0.575,High


## Top EVs for 2 leg bets

### Underdog picks

In [18]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 93 players...
Processing 84 players with valid predictions...
Generated 3294 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 125 combinations from 3294 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dillon Brooks,Lauri Markkanen,17.5,26.5,23.59,32.72,over,over,1,7.86,0.393,High,High
1,Zion Williamson,Lauri Markkanen,18.5,26.5,22.40,32.72,over,over,1,6.39,0.319,High,High
2,Harrison Barnes,Lauri Markkanen,9.5,26.5,12.72,32.72,over,over,0,5.90,0.295,High,High
3,Zion Williamson,Dillon Brooks,18.5,17.5,22.40,23.59,over,over,0,5.85,0.293,High,High
4,Day'Ron Sharpe,Dillon Brooks,6.5,17.5,9.00,23.59,over,over,0,5.17,0.258,Med,High


### Prizepicks picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 102 players...
Processing 92 players with valid predictions...
Generated 3942 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 137 combinations from 3942 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dillon Brooks,Lauri Markkanen,16.5,26.5,23.59,32.72,over,over,1,8.37,0.418,High,High
1,Nicolas Batum,Dillon Brooks,4.5,16.5,6.07,23.59,over,over,0,6.11,0.305,Med,High
2,Harrison Barnes,Dillon Brooks,9.5,16.5,12.72,23.59,over,over,0,6.03,0.301,High,High
3,Day'Ron Sharpe,Lauri Markkanen,6.5,26.5,9.00,32.72,over,over,0,5.90,0.295,Med,High
4,Zion Williamson,Lauri Markkanen,19.0,26.5,22.40,32.72,over,over,0,5.76,0.288,High,High


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 93 players...
Processing 86 players with valid predictions...
Generated 100898 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 57 combinations from 100898 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Anthony Black,Dillon Brooks,Lauri Markkanen,9.5,16.5,26.5,14.02,23.59,32.72,over,over,over,0,15.53,0.311,High,High,High
1,Goga Bitadze,Dillon Brooks,Lauri Markkanen,4.5,16.5,26.5,6.94,23.59,32.72,over,over,over,0,15.32,0.306,Med,High,High
2,Anthony Black,Goga Bitadze,Naji Marshall,9.5,4.5,10.5,14.02,6.94,14.54,over,over,over,0,10.45,0.209,High,Med,High
3,Zion Williamson,Naji Marshall,Dyson Daniels,18.5,10.5,12.5,22.40,14.54,8.79,over,over,under,0,8.90,0.178,High,High,Med
4,Harrison Barnes,Zion Williamson,Onyeka Okongwu,9.5,18.5,12.5,12.72,22.40,15.75,over,over,over,0,8.29,0.166,High,High,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.60, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 108 players...
Processing 98 players with valid predictions...
Generated 150000 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 65 combinations from 150000 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Dillon Brooks,Lauri Markkanen,Isaac Okoro,16.5,26.5,6.5,23.59,32.72,10.67,over,over,over,0,16.05,0.321,High,High,High
1,Anthony Black,Dillon Brooks,Lauri Markkanen,9.5,16.5,26.5,14.02,23.59,32.72,over,over,over,0,15.49,0.310,High,High,High
2,Anthony Black,Goga Bitadze,Isaac Okoro,9.5,4.5,6.5,14.02,6.94,10.67,over,over,over,0,11.25,0.225,High,Med,High
3,Goga Bitadze,Dyson Daniels,Nikola Vučević,4.5,12.5,18.5,6.94,8.79,13.84,over,under,under,0,9.43,0.189,Med,Med,High
4,Day'Ron Sharpe,Dyson Daniels,Nikola Vučević,6.5,12.5,18.5,9.00,8.79,13.84,over,under,under,0,8.68,0.174,Med,Med,High
